# Analisis Customer Churn - Telco Dataset

Proyek ini menganalisis data pelanggan telekomunikasi untuk memahami faktor-faktor yang mempengaruhi churn (pelanggan berhenti berlangganan) dan membangun model prediksi sederhana.

**Dataset:** Telco Customer Churn dari Kaggle
**Tujuan:** Mengidentifikasi pola churn dan membangun model klasifikasi

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

## 1. Memuat dan Memahami Data

In [4]:
df = pd.read_csv('telco_churn.csv')
print(f"Dataset: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"\nKolom:\n{list(df.columns)}")
df.head()

Dataset: 7043 baris, 21 kolom

Kolom:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
print("=== Informasi Dataset ===")
print(df.info())
print(f"\n=== Missing Values ===")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\n=== Distribusi Churn ===")
print(df['Churn'].value_counts())
print(f"\nPersentase Churn: {df['Churn'].value_counts(normalize=True)['Yes']*100:.1f}%")

=== Informasi Dataset ===
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-

## 2. Exploratory Data Analysis (EDA)

In [6]:
# Churn rate berdasarkan kontrak
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Chart 1: Churn by Contract
contract_churn = df.groupby(['Contract', 'Churn']).size().unstack()
contract_churn.plot(kind='bar', ax=axes[0], color=['#0F3057', '#00ABB3'])
axes[0].set_title('Churn berdasarkan Tipe Kontrak', fontweight='bold')
axes[0].set_xlabel('Tipe Kontrak')
axes[0].set_ylabel('Jumlah Pelanggan')
axes[0].tick_params(axis='x', rotation=45)

# Chart 2: Churn by Internet Service
internet_churn = df.groupby(['InternetService', 'Churn']).size().unstack()
internet_churn.plot(kind='bar', ax=axes[1], color=['#0F3057', '#00ABB3'])
axes[1].set_title('Churn berdasarkan Internet Service', fontweight='bold')
axes[1].set_xlabel('Internet Service')
axes[1].set_ylabel('Jumlah Pelanggan')
axes[1].tick_params(axis='x', rotation=45)

# Chart 3: Churn distribution
df['Churn'].value_counts().plot(kind='pie', ax=axes[2], autopct='%1.1f%%', colors=['#0F3057', '#00ABB3'], startangle=90)
axes[2].set_title('Distribusi Churn', fontweight='bold')
axes[2].set_ylabel('')

plt.tight_layout()
plt.savefig('eda_churn_overview.png', dpi=150)
plt.show()

In [7]:
# Distribusi tenure berdasarkan churn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram tenure
for label, color in [('No', '#0F3057'), ('Yes', '#00ABB3')]:
    subset = df[df['Churn'] == label]
    axes[0].hist(subset['tenure'], bins=30, alpha=0.7, label=f'Churn: {label}', color=color)
axes[0].set_title('Distribusi Tenure berdasarkan Churn', fontweight='bold')
axes[0].set_xlabel('Tenure (bulan)')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()

# Monthly charges
for label, color in [('No', '#0F3057'), ('Yes', '#00ABB3')]:
    subset = df[df['Churn'] == label]
    axes[1].hist(subset['MonthlyCharges'], bins=30, alpha=0.7, label=f'Churn: {label}', color=color)
axes[1].set_title('Distribusi Monthly Charges berdasarkan Churn', fontweight='bold')
axes[1].set_xlabel('Monthly Charges ($)')
axes[1].set_ylabel('Frekuensi')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_tenure_charges.png', dpi=150)
plt.show()

## 3. Insight dari EDA

Berdasarkan analisis visual di atas:
- **Kontrak bulanan** memiliki tingkat churn paling tinggi dibanding 1 atau 2 tahun
- **Pelanggan dengan tenure rendah** (baru berlangganan) lebih cenderung churn
- **Monthly charges tinggi** berkorelasi dengan tingkat churn yang lebih tinggi
- **Fiber optic** memiliki tingkat churn lebih tinggi dibanding DSL

## 4. Persiapan Data untuk Model

In [8]:
# Konversi TotalCharges ke numerik
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Encode target variable
df['Churn_encoded'] = df['Churn'].map({'No': 0, 'Yes': 1})

# Pilih fitur numerik
features = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Tambah fitur kategorikal yang di-encode
df['Contract_encoded'] = df['Contract'].map({'Month-to-month': 0, 'One year': 1, 'Two year': 2})
df['InternetService_encoded'] = df['InternetService'].map({'No': 0, 'DSL': 1, 'Fiber optic': 2})
features += ['Contract_encoded', 'InternetService_encoded']

X = df[features]
y = df['Churn_encoded']

print(f"Fitur yang digunakan: {features}")
print(f"Shape X: {X.shape}")
print(f"Distribusi target:\n{y.value_counts()}")

Fitur yang digunakan: ['tenure', 'MonthlyCharges', 'TotalCharges', 'Contract_encoded', 'InternetService_encoded']
Shape X: (7043, 5)
Distribusi target:
Churn_encoded
0    5174
1    1869
Name: count, dtype: int64


## 5. Membangun Model Prediksi

In [9]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Data training: {X_train.shape[0]} baris")
print(f"Data testing: {X_test.shape[0]} baris")

# Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Prediksi
y_pred = model.predict(X_test)

# Evaluasi
accuracy = accuracy_score(y_test, y_pred)
print(f"\nAkurasi model: {accuracy:.2%}")
print(f"\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Tidak Churn', 'Churn']))

Data training: 5634 baris
Data testing: 1409 baris

Akurasi model: 76.93%

=== Classification Report ===
              precision    recall  f1-score   support

 Tidak Churn       0.82      0.88      0.85      1036
       Churn       0.58      0.46      0.52       373

    accuracy                           0.77      1409
   macro avg       0.70      0.67      0.68      1409
weighted avg       0.76      0.77      0.76      1409



In [10]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Tidak Churn', 'Churn'],
            yticklabels=['Tidak Churn', 'Churn'])
axes[0].set_title('Confusion Matrix', fontweight='bold')
axes[0].set_xlabel('Prediksi')
axes[0].set_ylabel('Aktual')

# Feature importance
importances = pd.Series(model.feature_importances_, index=features).sort_values()
importances.plot(kind='barh', ax=axes[1], color='#0F3057')
axes[1].set_title('Feature Importance', fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('model_results.png', dpi=150)
plt.show()

In [11]:
# Churn rate per kategori
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Churn by gender
gender_churn = df.groupby(['gender', 'Churn']).size().unstack(fill_value=0)
gender_churn.plot(kind='bar', ax=axes[0, 0], color=['#0F3057', '#00ABB3'])
axes[0, 0].set_title('Churn berdasarkan Gender', fontweight='bold')
axes[0, 0].tick_params(axis='x', rotation=0)

# 2. Churn by Senior Citizen
senior_churn = df.groupby(['SeniorCitizen', 'Churn']).size().unstack(fill_value=0)
senior_churn.index = ['Non-Senior', 'Senior']
senior_churn.plot(kind='bar', ax=axes[0, 1], color=['#0F3057', '#00ABB3'])
axes[0, 1].set_title('Churn berdasarkan Senior Citizen', fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=0)

# 3. Churn by Payment Method
payment_churn = df.groupby(['PaymentMethod', 'Churn']).size().unstack(fill_value=0)
payment_churn.plot(kind='barh', ax=axes[1, 0], color=['#0F3057', '#00ABB3'])
axes[1, 0].set_title('Churn berdasarkan Metode Pembayaran', fontweight='bold')

# 4. Monthly charges vs Tenure scatter
colors = df['Churn'].map({'No': '#0F3057', 'Yes': '#00ABB3'})
axes[1, 1].scatter(df['tenure'], df['MonthlyCharges'], c=colors, alpha=0.3, s=10)
axes[1, 1].set_title('Tenure vs Monthly Charges', fontweight='bold')
axes[1, 1].set_xlabel('Tenure (bulan)')
axes[1, 1].set_ylabel('Monthly Charges ($)')

plt.tight_layout()
plt.savefig('eda_churn_detailed.png', dpi=150)
plt.show()

## Model Comparison

Membandingkan performa 3 algoritma machine learning untuk mencari model terbaik.

In [13]:
# Hapus NaN dari X dan y
X_train = X_train.fillna(X_train.median())
X_test = X_test.fillna(X_test.median())

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results.append({'Model': name, 'Accuracy': f'{acc:.2%}', 'F1-Score': f'{f1:.2%}'})
    print(f"{name}: Accuracy={acc:.2%}, F1-Score={f1:.2%}")

results_df = pd.DataFrame(results)
print("\n=== Perbandingan Model ===")
print(results_df.to_string(index=False))

Logistic Regression: Accuracy=80.20%, F1-Score=58.42%
Decision Tree: Accuracy=72.75%, F1-Score=47.11%
Random Forest: Accuracy=77.15%, F1-Score=52.37%

=== Perbandingan Model ===
              Model Accuracy F1-Score
Logistic Regression   80.20%   58.42%
      Decision Tree   72.75%   47.11%
      Random Forest   77.15%   52.37%


In [15]:
# Visualisasi perbandingan model
fig, ax = plt.subplots(figsize=(10, 5))

model_names = [r['Model'] for r in results]
accuracies = [float(r['Accuracy'].strip('%'))/100 for r in results]
f1_scores = [float(r['F1-Score'].strip('%'))/100 for r in results]

x = range(len(model_names))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], accuracies, width, label='Accuracy', color='#0F3057')
bars2 = ax.bar([i + width/2 for i in x], f1_scores, width, label='F1-Score', color='#00ABB3')

ax.set_ylabel('Score')
ax.set_title('Perbandingan Performa Model', fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend()
ax.set_ylim(0, 1)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

**Insight:** Random Forest memberikan performa terbaik secara keseluruhan dengan accuracy dan F1-Score tertinggi. Logistic Regression memberikan hasil yang kompetitif meskipun lebih sederhana. Decision Tree cenderung overfitting sehingga performanya lebih rendah.

## 6. Kesimpulan dan Rekomendasi Bisnis

### Temuan Utama:
1. **Tenure** adalah faktor terpenting dalam memprediksi churn - pelanggan baru lebih berisiko
2. **Monthly Charges** berpengaruh signifikan - tagihan tinggi meningkatkan risiko churn
3. **Tipe kontrak** sangat berpengaruh - kontrak bulanan paling rentang churn

### Rekomendasi:
1. Buat program loyalitas untuk pelanggan baru (0-12 bulan pertama) berupa diskon atau benefit tambahan
2. Tawarkan insentif migrasi dari kontrak bulanan ke kontrak tahunan
3. Review pricing strategy untuk pelanggan dengan monthly charges tinggi
4. Fokuskan retensi pada segment fiber optic yang memiliki churn rate tinggi